# Exploratory GMM Clustering Notebook

This notebook mirrors the src pipeline but adds an exploratory switch to force a fixed number of clusters.

Use the settings cell to temporarily hardcode `k=3` (or another value) and regenerate `cluster_phase_heatmap.png`.

In [1]:
from pathlib import Path
import sys
import json

# Resolve repository root and ensure src is importable from this notebook.
cwd = Path.cwd()
if (cwd / "src").exists():
    REPO_ROOT = cwd
elif (cwd.parent / "src").exists():
    REPO_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate repository root containing src/")

src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.append(src_path)

from physio_pipeline.config import PipelineConfig
from physio_pipeline.data_io import ensure_output_directories, load_dataset
from physio_pipeline.preprocessing import preprocess_dataset
from physio_pipeline.pca_stage import run_pca
from physio_pipeline.model_selection import evaluate_gmm_candidates
from physio_pipeline.clustering import fit_final_gmm
from physio_pipeline.evaluation import evaluate_phase_alignment
from physio_pipeline.visualization import (
    plot_cumulative_variance,
    plot_aic_bic,
    plot_cluster_phase_heatmap,
)

REPO_ROOT

WindowsPath('c:/Users/Damsgaard/Documents/Github/02582_CDA_Case2')

## Exploration Settings

Change values in this cell to explore different clustering assumptions.

- Set `USE_FIXED_K = True` and `FIXED_K = 3` to hardcode three clusters.
- Set `USE_FIXED_K = False` to use BIC-selected k.

In [ ]:
DATA_PATH = REPO_ROOT / "HR_data_2" / "HR_data_2.csv"
OUTPUT_DIR = REPO_ROOT / "outputs"

PCA_VARIANCE_THRESHOLD = 0.80
K_MIN = 1
K_MAX = 8
RANDOM_STATE = 42
GMM_N_INIT = 10
GMM_COVARIANCE_TYPE = "diag"

# Exploratory switch: set True to hardcode k, False to use BIC minimum.
USE_FIXED_K = True
FIXED_K = 3

config = PipelineConfig(
    data_path=DATA_PATH,
    output_dir=OUTPUT_DIR,
    pca_variance_threshold=PCA_VARIANCE_THRESHOLD,
    gmm_k_min=K_MIN,
    gmm_k_max=K_MAX,
    random_state=RANDOM_STATE,
    gmm_n_init=GMM_N_INIT,
    gmm_covariance_type=GMM_COVARIANCE_TYPE,
)

if USE_FIXED_K and not (K_MIN <= FIXED_K <= K_MAX):
    raise ValueError(f"FIXED_K must be in [{K_MIN}, {K_MAX}]")

config

PipelineConfig(data_path=WindowsPath('c:/Users/Damsgaard/Documents/Github/02582_CDA_Case2/HR_data_2/HR_data_2.csv'), output_dir=WindowsPath('c:/Users/Damsgaard/Documents/Github/02582_CDA_Case2/outputs'), random_state=42, pca_variance_threshold=0.8, gmm_k_min=1, gmm_k_max=8, gmm_n_init=10, gmm_covariance_type='diag', missingness_iterative_threshold=0.2, group_column='Individual', phase_column='Phase', cluster_column='Cluster', required_columns=('Individual', 'Phase', 'Round'), physiological_prefixes=('HR_', 'EDA_', 'TEMP_'))

In [3]:
# 1) Data loading and preprocessing
figures_dir, results_dir = ensure_output_directories(config.output_dir)
raw_df = load_dataset(config.data_path, config.required_columns)

prep = preprocess_dataset(
    df=raw_df,
    prefixes=config.physiological_prefixes,
    group_column=config.group_column,
    missingness_iterative_threshold=config.missingness_iterative_threshold,
    random_state=config.random_state,
)

feature_matrix = prep.processed_df[prep.feature_columns].to_numpy()
prep.missingness_report.to_csv(results_dir / "missingness_report.csv", index=False)

print(f"Rows: {len(raw_df)}")
print(f"Physiological feature count: {len(prep.feature_columns)}")
print(f"Imputation method used: {prep.imputation_method}")
print(f"Global missingness ratio: {prep.global_missingness_ratio:.6f}")

Rows: 312
Physiological feature count: 51
Imputation method used: median
Global missingness ratio: 0.000126


In [4]:
# 2) PCA diagnostics and transformation
pca_result = run_pca(
    feature_matrix=feature_matrix,
    variance_threshold=config.pca_variance_threshold,
    random_state=config.random_state,
)

pca_result.variance_table.to_csv(results_dir / "pca_variance_table.csv", index=False)
plot_cumulative_variance(
    variance_table=pca_result.variance_table,
    variance_threshold=config.pca_variance_threshold,
    output_path=figures_dir / "pca_cumulative_explained_variance.png",
)

print(f"Selected PCA components (>= {config.pca_variance_threshold:.0%} variance): {pca_result.selected_components}")
pca_result.variance_table.head(10)

Selected PCA components (>= 80% variance): 14


,component,explained_variance_ratio,cumulative_explained_variance
0,1,0.202623,0.202623
1,2,0.090646,0.293268
2,3,0.081332,0.374600
3,4,0.073221,0.447821
4,5,0.066036,0.513857
5,6,0.054504,0.568361
6,7,0.051223,0.619584
7,8,0.034922,0.654506
8,9,0.030937,0.685443
9,10,0.029549,0.714991


In [5]:
# 3) GMM model selection with AIC/BIC for k=1..8
selection = evaluate_gmm_candidates(
    feature_matrix=pca_result.transformed,
    k_min=config.gmm_k_min,
    k_max=config.gmm_k_max,
    random_state=config.random_state,
    n_init=config.gmm_n_init,
    covariance_type=config.gmm_covariance_type,
)

selection.scores.to_csv(results_dir / "gmm_aic_bic_scores.csv", index=False)
plot_aic_bic(selection.scores, figures_dir / "gmm_aic_bic_by_k.png")

bic_selected_k = int(selection.optimal_k)
print(f"BIC-selected k: {bic_selected_k}")
selection.scores

BIC-selected k: 2


,k,aic,bic,log_likelihood
0,1,16228.122353,16332.926443,-25.916863
1,2,16080.685753,16294.036935,-25.587637
2,3,16055.164125,16377.062399,-25.453789
3,4,15882.558340,16313.003707,-25.084228
4,5,16029.400846,16568.393305,-25.226604
5,6,16008.497982,16656.037533,-25.100157
6,7,16012.179551,16768.266195,-25.013108
7,8,16004.528662,16869.162399,-24.907898


In [8]:
# 4) Final clustering with optional fixed-k override
if USE_FIXED_K:
    final_k = int(FIXED_K)
    print(f"Using fixed exploratory k={final_k}")
else:
    final_k = bic_selected_k
    print(f"Using BIC-selected k={final_k}")

_, labels = fit_final_gmm(
    feature_matrix=pca_result.transformed,
    n_components=final_k,
    random_state=config.random_state,
    n_init=config.gmm_n_init,
    covariance_type=config.gmm_covariance_type,
)

enriched_df = raw_df.copy()
enriched_df[config.cluster_column] = labels
evaluation = evaluate_phase_alignment(
    df=enriched_df,
    cluster_column=config.cluster_column,
    phase_column=config.phase_column,
)

suffix = f"_k{final_k}"

heatmap_path = figures_dir / f"cluster_phase_heatmap{suffix}.png"
plot_cluster_phase_heatmap(evaluation.normalized_table, heatmap_path)

enriched_df.to_csv(results_dir / f"hr_data_with_clusters{suffix}.csv", index=False)
evaluation.counts_table.to_csv(results_dir / f"cluster_phase_counts{suffix}.csv")
evaluation.normalized_table.to_csv(results_dir / f"cluster_phase_normalized{suffix}.csv")

with (results_dir / f"mapped_classification_report{suffix}.txt").open("w", encoding="utf-8") as file:
    file.write(evaluation.mapped_report)

with (results_dir / f"cluster_to_phase_mapping{suffix}.json").open("w", encoding="utf-8") as file:
    json.dump(evaluation.cluster_to_phase_map, file, indent=2)

selected_row = selection.scores[selection.scores["k"] == final_k].iloc[0]
summary = {
    "use_fixed_k": bool(USE_FIXED_K),
    "fixed_k": int(FIXED_K) if USE_FIXED_K else None,
    "bic_selected_k": int(bic_selected_k),
    "final_k_used": int(final_k),
    "selected_k_aic": float(selected_row["aic"]),
    "selected_k_bic": float(selected_row["bic"]),
    "nmi": float(evaluation.nmi),
    "selected_pca_components": int(pca_result.selected_components),
    "global_missingness_ratio": float(prep.global_missingness_ratio),
}

with (results_dir / f"metrics_summary{suffix}.json").open("w", encoding="utf-8") as file:
    json.dump(summary, file, indent=2)

print(f"Saved exploratory heatmap to: {heatmap_path}")
print(f"NMI (Phase vs Cluster): {evaluation.nmi:.4f}")
print("Saved exploratory outputs with suffix:", suffix)
evaluation.normalized_table

Using fixed exploratory k=3
Saved exploratory heatmap to: c:\Users\Damsgaard\Documents\Github\02582_CDA_Case2\outputs\figures\cluster_phase_heatmap_k3.png
NMI (Phase vs Cluster): 0.0209
Saved exploratory outputs with suffix: _k3


Phase,phase1,phase2,phase3
Cluster,,,
0,0.468354,0.253165,0.278481
1,0.355932,0.313559,0.330508
2,0.217391,0.408696,0.373913
